In [1]:
from condastats.cli import overall, data_source
import pypistats
from datetime import date
from dateutil.relativedelta import relativedelta
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
import conda.cli.python_api

## Packages

In [3]:
stdout, stderr, return_code = conda.cli.python_api.run_command(
    "search", "-c", "conda-forge", "openff-*"
)
openff_packages = set()
for line in stdout.split("\n"):
    if line.strip():
        field = line.split()[0].strip()
        if field.startswith("openff-"):
            openff_packages.add(field)

In [4]:
search_packages = []
# remove non-base packages
for package in sorted(openff_packages):
    if f"{package}-base" in openff_packages:
        continue
    search_packages.append(package)

search_packages = sorted(search_packages)
print(search_packages)

['openff-amber-ff-ports', 'openff-bespokefit', 'openff-evaluator-base', 'openff-forcefields', 'openff-fragmenter-base', 'openff-interchange-base', 'openff-models', 'openff-nagl-base', 'openff-nagl-models', 'openff-qcsubmit', 'openff-recharge', 'openff-toolkit-base', 'openff-toolkit-examples', 'openff-units', 'openff-utilities']


# Monthly downloads

## conda-forge downloads

In [5]:
# Get historical conda-forge downloads
# This usually lags by two months so we don't have to worry about
# incomplete months

condadf = overall(
    search_packages,
    data_source='conda-forge',
    monthly=True
)
condadf[-5:]

pkg_name       time   
openff-models  2023-06    28
               2023-07    29
               2023-08    41
               2023-09    16
               2023-10    15
Name: counts, dtype: int64

In [6]:
df = pd.DataFrame(condadf).reset_index()

In [7]:
allowed_months = [f"2023-{x:02d}" for x in range(6, 11)]
past_five_months = df[df.time.isin(allowed_months)]

In [8]:
average = (past_five_months.counts.sum() / len(past_five_months.time.unique()))
average

21494.4

# Dependants

## conda-forge dependants

In [9]:
import tqdm
import subprocess

dependants = set()
for package in tqdm.tqdm(openff_packages):
    toggle = False
    proc = subprocess.run(
        f"mamba repoquery whoneeds {package} -c conda-forge --quiet",
        shell=True,
        capture_output=True,
        text=True,
    )
    for line in proc.stdout.split("\n"):
        if line.strip().startswith("Name"):
            toggle = True
            continue
        if toggle:
            fields = line.strip().split()
            if len(fields) > 1:
                dependants.add(fields[0])

100%|███████████████████████████████████████████| 20/20 [00:16<00:00,  1.19it/s]


In [11]:
non_openff_dependants = [
    x for x in dependants if not x.startswith("openff-")
]

In [12]:
non_openff_dependants

['absolv',
 'perses',
 'absolv-base',
 'gufe',
 'paprika',
 'smee-base',
 'de-forcefields',
 'qubekit',
 'fegrow',
 'espaloma',
 'openfe',
 'nagl',
 'cinnabar',
 'smee',
 'openmmforcefields',
 'openfe-analysis',
 'smirnoff-plugins']

In [33]:
len(non_openff_dependants)

17

## GitHub

In [13]:
openff_packages

{'openff-amber-ff-ports',
 'openff-bespokefit',
 'openff-evaluator',
 'openff-evaluator-base',
 'openff-forcefields',
 'openff-fragmenter',
 'openff-fragmenter-base',
 'openff-interchange',
 'openff-interchange-base',
 'openff-models',
 'openff-nagl',
 'openff-nagl-base',
 'openff-nagl-models',
 'openff-qcsubmit',
 'openff-recharge',
 'openff-toolkit',
 'openff-toolkit-base',
 'openff-toolkit-examples',
 'openff-units',
 'openff-utilities'}

In [15]:
import json

all_public_repos = set()
n_private_dependants = 0
for package in tqdm.tqdm(openff_packages):
    if package.endswith("-base"):
        proc = subprocess.run(
            f'github-dependents-info --repo openforcefield/{package} --sort name --json',
            capture_output=True,
            text=True,
            shell=True,
        )
        contents = json.loads(proc.stdout)
        public_repos = [
            x["name"]
            for x in contents["all_public_dependent_repos"]
        ]
        all_public_repos |= set(public_repos)
        n_private_dependants += contents["private_dependents_number"]
        


100%|███████████████████████████████████████████| 20/20 [00:04<00:00,  4.63it/s]


In [18]:
all_public_repos

set()

In [17]:
n_private_dependants

0

In [1]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from github import Github

%matplotlib inline


In [2]:
git = Github(os.environ['GH_API_TOKEN'])

In [3]:
org = "openforcefield"

In [4]:
all_repos = [
    repo.name for repo in git.get_user(org).get_repos()
]

In [5]:
all_repos

['2021-benchmarking-workshop',
 '2021-bespokefit-workshop',
 '2023-workshop-vignettes',
 'alchemiscale',
 'alchemiscale-fah',
 'amber-ff-porting',
 'bayes-implicit-solvent',
 'best-practices-observables',
 'ccpbiosim-2023',
 'cheminformatics-toolkit-equivalence',
 'cmiles',
 'CMILES-Cloud',
 'dangerbot',
 'discussions',
 'interchange-regression-testing',
 'interchange-workshop-2022',
 'MiniDrugBank',
 'MPID_plugin',
 'nistdataselection',
 'open-forcefield-data',
 'open-forcefield-group',
 'open-forcefield-tools',
 'openff-amber-ff-ports',
 'openff-benchmark',
 'openff-bespokefit',
 'openff-cli',
 'openff-docs',
 'openff-evaluator',
 'openff-forcebalance',
 'openff-forcefields',
 'openff-fragmenter',
 'openff-gopt',
 'openff-interchange',
 'openff-models',
 'openff-nagl',
 'openff-nagl-models',
 'openff-qcsubmit',
 'openff-recharge',
 'openff-reference',
 'openff-sage',
 'openff-sphinx-theme',
 'openff-toolkit',
 'openff-units',
 'openff-utilities',
 'openforcefield-forcebalance',
 'ope

In [8]:
all_counts = 0

for repo_name in all_repos:
    repo = git.get_repo(f"{org}/{repo_name}")
    contents = repo.get_views_traffic(per="week")
    last_fortnight = contents["views"]
    counts = sum([x.count for x in last_fortnight])
    all_counts += counts

In [9]:
all_counts

6549

In [10]:
all_unique_counts = 0

for repo_name in all_repos:
    repo = git.get_repo(f"{org}/{repo_name}")
    contents = repo.get_views_traffic(per="week")
    last_fortnight = contents["views"]
    counts = sum([x.uniques for x in last_fortnight])
    all_unique_counts += counts
all_unique_counts

941